# Verify preserved frozen factor-probe artifact

Attach **Notebook Output Files** from `thestonedape/freeze`, exact saved version `335849368`. This notebook does not refit probes or access raw EEG. It re-hashes the preserved result, checks the frozen protocol and admission decisions, and emits a small verification report. Enable Internet and the private Kaggle secret `GITHUB_TOKEN`.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
VERIFIER_COMMIT = '22d135af01e3daeb29f6ff4580d5d49097e0af7d'
PRESERVED_SOURCE_ID = 'kaggle-code-thestonedape-freeze-scriptVersionId-335849368'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-aware-eeg2text-factor-probe-verification'
assert len(VERIFIER_COMMIT) == 40 and PRESERVED_SOURCE_ID.endswith('335849368')


In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', VERIFIER_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == VERIFIER_COMMIT
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'test_verify_frozen_factor_probe_artifact.py')], check=True)
print({'python': platform.python_version(), 'verifier_commit': actual_commit, 'regression': 'PASS'})


In [ ]:
manifest_candidates = glob.glob('/kaggle/input/**/probe_manifest.json', recursive=True)
artifact_roots = []
for manifest_path in manifest_candidates:
    candidate = os.path.dirname(manifest_path)
    required = ['run_metadata.json', 'factor_admission.csv', 'planned_contrasts.csv', 'probe_metrics.csv', 'probe_predictions.csv', 'probe_selection.csv', 'frozen_protocol']
    if all(os.path.exists(os.path.join(candidate, name)) for name in required):
        artifact_roots.append(candidate)
assert len(artifact_roots) == 1, ('Attach exactly one complete saved factor-probe output', artifact_roots, manifest_candidates)
ARTIFACT_ROOT = artifact_roots[0]
print({'preserved_source_id': PRESERVED_SOURCE_ID, 'artifact_root': ARTIFACT_ROOT, 'top_level_files': sorted(os.listdir(ARTIFACT_ROOT))})


In [ ]:
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
os.makedirs(OUTPUT)
report_path = os.path.join(OUTPUT, 'verification_report.json')
subprocess.run([
    sys.executable, os.path.join(WORKTREE, 'evaluation', 'verify_frozen_factor_probe_artifact.py'),
    '--artifact-root', ARTIFACT_ROOT, '--output-report', report_path,
    '--preserved-source-id', PRESERVED_SOURCE_ID,
], check=True)
report = json.load(open(report_path, encoding='utf-8'))
assert report['status'] == 'pass'
assert report['preserved_source_id'] == PRESERVED_SOURCE_ID
assert report['admitted_factors'] == ['tsr_instruction_relation']
assert report['checks'] == {
    'all_declared_hashes_revalidated': True,
    'frozen_protocol_revalidated': True,
    'held_out_test_accessed': False,
    'admission_decisions_match_frozen_log': True,
}
verification_metadata = {
    'status': 'pass', 'verifier_commit': actual_commit,
    'preserved_source_id': PRESERVED_SOURCE_ID,
    'probe_manifest_sha256': report['probe_manifest_sha256'],
    'verification_report_sha256_pending': True, 'test_accessed': False,
}
with open(os.path.join(OUTPUT, 'verification_run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(verification_metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')


In [ ]:
def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()
report_sha256 = digest(report_path)
metadata_path = os.path.join(OUTPUT, 'verification_run_metadata.json')
verification_metadata = json.load(open(metadata_path, encoding='utf-8'))
verification_metadata['verification_report_sha256'] = report_sha256
verification_metadata.pop('verification_report_sha256_pending')
with open(metadata_path, 'w', encoding='utf-8') as handle:
    json.dump(verification_metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')
print({'status': report['status'], 'preserved_source_id': report['preserved_source_id'], 'admitted_factors': report['admitted_factors'], 'null_factors': report['null_factors'], 'probe_manifest_sha256': report['probe_manifest_sha256'], 'verification_report_sha256': report_sha256})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
print('FROZEN FACTOR PROBE PRESERVED-ARTIFACT VERIFICATION: PASS')
